In [53]:
# importing libraries

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.losses import BinaryCrossentropy

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
train_df = pd.read_pickle('data/intermediate_files/ranker_train_df.pkl')
val_df = pd.read_pickle('data/intermediate_files/ranker_validation_df.pkl')
test_df = pd.read_pickle('data/intermediate_files/ranker_test_df.pkl')

### Preprocessing

##### 1. Deterministic features

In [ ]:
def preprocess_pre_sample(df):
    tp_df = df.copy()

    # Time features
    tp_df["query_time"] = pd.to_datetime(tp_df["query_time"])
    tp_df["day"] = tp_df["query_time"].dt.day_name().astype(str)
    tp_df["hour_of_day"] = tp_df["query_time"].dt.hour.astype(str)

    # Log features for skewed numeric columns
    for col in ["past_view_count", "past_atc_count", "past_order_count", "cg_score_raw"]:
        tp_df[col] = tp_df[col].fillna(0)
        tp_df[f"log_{col}"] = np.log1p(tp_df[col].astype(float))

    # Basic null handling
    tp_df["cg_score_norm"] = tp_df["cg_score_norm"].fillna(0)
    tp_df["user_history_bucket"] = tp_df["user_history_bucket"].fillna("unknown").astype(str)

    # Binary columns
    binary_base_cols = [
        "candidate_previously_purchased", "candidate_in_last_k_items",
        "from_user_co_interaction", "from_user_category", "from_user_generality"
    ]
    for col in binary_base_cols:
        if col in tp_df.columns:
            tp_df[col] = tp_df[col].fillna(0).astype("int8")

    # Rank bucket flags; NaN rank becomes 0 for every bucket
    rank_cols = ["cg_rank", "user_co_interaction_rank", "user_category_rank", "user_generality_rank"]
    bucket_defs = {
        "1_10": (1, 10),
        "11_50": (11, 50),
        "51_100": (51, 100),
        "101_250": (101, 250),
        "251_500": (251, 500),
    }

    for col in rank_cols:
        if col not in tp_df.columns:
            continue
        for bucket_name, (lo, hi) in bucket_defs.items():
            tp_df[f"{col}_bucket_{bucket_name}"] = tp_df[col].between(lo, hi).fillna(False).astype("int8")

    return tp_df

##### 2. Negative Sampling

In [ ]:

def sample_negatives_by_query(df, label_col="label_engagement", query_col="global_query_id", neg_per_pos=10, random_state=42):
    sampled_groups = []
    rng = np.random.default_rng(random_state)

    for _, group in df.groupby(query_col):
        pos = group[group[label_col] == 1]
        neg = group[group[label_col] == 0]

        # Skip all-negative queries for first ranker training
        if len(pos) == 0:
            continue

        n_neg = min(len(neg), len(pos) * neg_per_pos)

        if n_neg > 0:
            neg_sample = neg.sample(n=n_neg, random_state=int(rng.integers(0, 1_000_000)))
            sampled_groups.append(pd.concat([pos, neg_sample], axis=0))
        else:
            sampled_groups.append(pos)

    sampled_df = pd.concat(sampled_groups, ignore_index=True)
    sampled_df = sampled_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return sampled_df

##### 3. Sanity Cleaning

In [ ]:
def clean_deep_features(df, dense_features, cat_features):
    df = df.copy()

    for col in dense_features:
        df[col] = df[col].fillna(0).astype("float32")

    for col in cat_features:
        df[col] = df[col].fillna('unknown').astype(str)

    return df

In [ ]:
# sampling validation/test to make it smaller, might comment this later

def sample_query_groups(df, n_queries, query_col='global_query_id', random_state=42):

    query_ids = df[query_col].drop_duplicates()
    sample_ids = query_ids.sample(n=min(n_queries, len(query_ids)), random_state=random_state)

    return df[df[query_col].isin(sample_ids)].copy()

### Tensor Flow Data Set Creation

In [ ]:
def make_tf_dataset(df, dense_feature_cols, cat_feature_cols, label_col, 
                    batch_size=512, shuffle=False, random_state=42):

    feature_dict = {}
    
    feature_dict['dense_features'] = df[dense_feature_cols].astype('float32').values
    
    # Separate categorical inputs
    for col in cat_feature_cols:
        feature_dict[col] = df[col].astype(str).values
    
    labels = df[label_col].astype('float32').values

    # creates tensorflow training input
    ds = tf.data.Dataset.from_tensor_slices((feature_dict, labels))

    # shuffling the dataset using a rolling window of 1L rows at a time 
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(df), 1_00_000), seed=random_state)

    # prepares data in form of batches, and also prefetch(tf.data.AUTOTUNE) improves 
    # performance by preparing the next training batch while the current training is happening
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds

### Loopup layers

##### Item Vocabulary

In [ ]:
def build_item_vocab(train_df, item_col="candidate_itemid", 
                     min_freq=2, max_items=50000):
    item_counts = train_df[item_col].astype(str).value_counts()

    # value counts result in an descending sorted output, so head(max_items) will return the top 50k items based on count
    item_vocab = (
        item_counts[item_counts >= min_freq]
        .head(max_items)
        .index
        .tolist()
    )

    return item_vocab


In [ ]:
def build_lookup_layers(train_df, embedding_cat_cols):

    lookup_layers = {}
    item_vocab = None

    for col in embedding_cat_cols:
        if col == 'candidate_itemid':
            vocab = build_item_vocab(train_df, item_col=col)
            item_vocab = vocab.copy()
        else:
            vocab = train_df[col].astype(str).unique().tolist()

        lookup_layers[col] = layers.StringLookup(
            vocabulary=vocab, mask_token=None, 
            num_oov_indices=1, output_mode="int", name=f'{col}_lookup'
        )

    return lookup_layers, item_vocab

##### Coverage Check

In [ ]:
def item_vocab_coverage(df, item_vocab, item_col="candidate_itemid", label_col="label_engagement"):
    vocab_set = set(item_vocab)
    item_str = df[item_col].astype(str)

    row_coverage = item_str.isin(vocab_set).mean()

    pos_df = df[df[label_col] == 1]
    if len(pos_df) > 0:
        pos_coverage = pos_df[item_col].astype(str).isin(vocab_set).mean()
    else:
        pos_coverage = np.nan

    return {
        "row_coverage": row_coverage,
        "positive_row_coverage": pos_coverage,
        "n_vocab_items": len(item_vocab),
        "n_unique_items_in_df": item_str.nunique(),
    }


# check for both train and val
# row_coverage = % of candidate rows whose item has its own embedding
# positive_row_coverage = % of positive rows whose item has its own embedding


### Model Build

In [ ]:
### Deep Ranker v2

def build_deep_ranker_v2(dense_input_dim, embedding_cat_cols, lookup_layers):

    # input prep
    inputs = {}
    encoded_parts = []
    
    #prepare dense/binary inputs
    dense_input = layers.Input(shape=(dense_input_dim,), name='dense_features')
    inputs['dense_features'] = dense_input    
        
    # normalizing dense input
    x_dense = layers.BatchNormalization(name='dense_input_norm')(dense_input)
    encoded_parts.append(x_dense)

    # embedding inputs
    for col in embedding_cat_cols:
        cat_input = layers.Input(shape=(), dtype=tf.string, name=col)
        inputs[col] = cat_input

        lookup = lookup_layers[col]
        vocab_size = lookup.vocabulary_size()

        # embedding size for later use
        if col in 'candidate_itemid':
            emb_dim = 8
        elif col in ['hour_of_day', 'day', 'user_history_bucket']:
            emb_dim = 4
        else:
            emb_dim = 4

        x = lookup(cat_input)

        x = layers.Embedding(input_dim=vocab_size,
            output_dim=emb_dim,
            name=f"{col}_embedding")(x)

        x = layers.Flatten(name=f'{col}_flatten')(x)
        encoded_parts.append(x)

    # concatenate dense features and embeddings
    x = layers.Concatenate(name='concat_features')(encoded_parts)    

    # model prep
    # hidden layers - num of hidden layers to be found out using val df
    x = layers.Dense(units=128, activation='relu', name='layer1')(x)

    x = layers.Dense(units=64, activation='relu', name='layer2')(x)
    
    x = layers.Dense(units=32, activation='relu', name='layer3')(x)
    # output layer
    output = layers.Dense(units=1, activation='linear', name="engagement_score")(x)

    # combining all to form the model
    model = Model(inputs=inputs, outputs=output)    

    return model


### Train Data & Validation data Creation

In [ ]:
## creating column name lists
label_col = 'label_engagement'

continuous_cols = ["log_past_view_count", "log_past_atc_count", "log_past_order_count", "log_cg_score_raw", "cg_score_norm",]

categorical_cols = ["candidate_itemid", "user_history_bucket", "day", "hour_of_day"]

binary_cols = ["candidate_previously_purchased", "candidate_in_last_k_items", "from_user_co_interaction", "from_user_category",
    "from_user_generality"]

rank_bucket_cols = [
    f"{rank_col}_bucket_{bucket}"
    for rank_col in ["cg_rank", "user_co_interaction_rank", "user_category_rank", "user_generality_rank"]
    for bucket in ["1_10", "11_50", "51_100", "101_250", "251_500"]
]

deep_cat_cols = categorical_cols
deep_dense_cols = continuous_cols+binary_cols+rank_bucket_cols

In [ ]:
# train DS

neg_per_pos = 10
random_state = 42

# Deterministic features first
train_pre = preprocess_pre_sample(train_df)

# Negative sampling only on train
train_sampled = sample_negatives_by_query(
    train_pre,
    label_col=label_col,
    query_col="global_query_id",
    neg_per_pos=neg_per_pos,
    random_state=random_state
)

# getting all the relevant columns
dense_feature_cols = [c for c in deep_dense_cols if c in train_sampled.columns]
cat_feature_cols = [c for c in deep_cat_cols if c in train_sampled.columns]

# final cleaning
train_model_df = clean_deep_features(train_sampled, dense_feature_cols, cat_feature_cols)

# creating tenssorflow datasets
train_ds = make_tf_dataset(train_model_df, dense_feature_cols, cat_feature_cols, label_col, shuffle=True)

In [ ]:
# Sanity checks:
print("Train shape:", train_model_df.shape)
print("Positive rate:", train_model_df[label_col].mean())
print("Dense feature count:", len(dense_feature_cols))
print("Embedding categorical cols:", cat_feature_cols)
print("Nulls in dense features:", train_model_df[dense_feature_cols].isna().sum().sum())


Train shape: (60577, 55)
Positive rate: 0.09090909090909091
Dense feature count: 30
Embedding categorical cols: ['candidate_itemid', 'user_history_bucket', 'day', 'hour_of_day']
Nulls in dense features: 0


In [ ]:
# Val DS

# sample queries from validation df
val_small = sample_query_groups(val_df, 10000, query_col='global_query_id', random_state=42)

# Deterministic features first
val_pre = preprocess_pre_sample(val_small)

# final cleaning
val_model_df = clean_deep_features(val_pre, dense_feature_cols, cat_feature_cols)

# creating tenssorflow datasets
val_ds = make_tf_dataset(val_model_df, dense_feature_cols, cat_feature_cols, label_col, shuffle=False)


### Model initalisation

In [ ]:
# creating lookup
lookup_layers, item_vocab = build_lookup_layers(train_model_df, cat_feature_cols)

# checking coverage
print("Train item coverage:")
print(item_vocab_coverage(train_model_df, item_vocab))

print("Validation item coverage:")
print(item_vocab_coverage(val_model_df, item_vocab))

Train item coverage:
{'row_coverage': np.float64(0.870445878798884), 'positive_row_coverage': np.float64(0.8523697112765571), 'n_vocab_items': 5474, 'n_unique_items_in_df': 13322}
Validation item coverage:
{'row_coverage': np.float64(0.671561653009626), 'positive_row_coverage': np.float64(0.5910931174089069), 'n_vocab_items': 5474, 'n_unique_items_in_df': 39527}


In [ ]:
# building the model

dnn_model_v2 = build_deep_ranker_v2(
    dense_input_dim=len(dense_feature_cols), 
    embedding_cat_cols=cat_feature_cols, 
    lookup_layers=lookup_layers
    )

dnn_model_v2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=BinaryCrossentropy(from_logits=True), 
    metrics=[
        tf.keras.metrics.AUC(name='roc_auc', from_logits=True),
        tf.keras.metrics.AUC(curve='PR', name='pr_auc', from_logits=True)
    ]
    )

dnn_model_v2.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ candidate_itemid    │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_history_bucket │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ day (InputLayer)    │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hour_of_day         │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ candidate_itemid_l… │ (None)            │          0 │ candidate_itemid… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_history_bucke… │ (None)            │          0 │ user_history_buc… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ day_lookup          │ (None)            │          0 │ day[0][0]         │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hour_of_day_lookup  │ (None)            │          0 │ hour_of_day[0][0] │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_features      │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ candidate_itemid_e… │ (None, 32)        │    175,200 │ candidate_itemid… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_history_bucke… │ (None, 4)         │         24 │ user_history_buc… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ day_embedding       │ (None, 4)         │         32 │ day_lookup[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hour_of_day_embedd… │ (None, 4)         │        100 │ hour_of_day_look… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_input_norm    │ (None, 30)        │        120 │ dense_features[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ candidate_itemid_f… │ (None, 32)        │          0 │ candidate_itemid… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_history_bucke… │ (None, 4)         │          0 │ user_history_buc… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ day_flatten         │ (None, 4)         │          0 │ day_embedding[0]… │
│ (Flatten)           │                   │            │                 

 Total params: 195,445 (763.46 KB)

 Trainable params: 195,385 (763.22 KB)

 Non-trainable params: 60 (240.00 B)

##### Fitting the model

In [ ]:
trial_model_history = dnn_model_v2.fit(train_ds, epochs=20, validation_data=val_ds)

Epoch 1/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 22s 149ms/step - loss: 0.1792 - pr_auc: 0.6731 - roc_auc: 0.8969 - val_loss: 0.0833 - val_pr_auc: 0.0111 - val_roc_auc: 0.9230
Epoch 2/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 17s 142ms/step - loss: 0.1327 - pr_auc: 0.7914 - roc_auc: 0.9444 - val_loss: 0.0660 - val_pr_auc: 0.0108 - val_roc_auc: 0.9213
Epoch 3/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 17s 142ms/step - loss: 0.1182 - pr_auc: 0.8246 - roc_auc: 0.9611 - val_loss: 0.0432 - val_pr_auc: 0.0101 - val_roc_auc: 0.9080
Epoch 4/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 17s 143ms/step - loss: 0.1087 - pr_auc: 0.8450 - roc_auc: 0.9685 - val_loss: 0.0432 - val_pr_auc: 0.0096 - val_roc_auc: 0.8956
Epoch 5/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 17s 140ms/step - loss: 0.1027 - pr_auc: 0.8597 - roc_auc: 0.9725 - val_loss: 0.0544 - val_pr_auc: 0.0089 - val_roc_auc: 0.9010
Epoch 6/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 17s 143ms/step - loss: 0.0970 - pr_auc: 0.8732 - roc_auc: 0.9754 - val_loss: 0.0492 - val_pr_auc: 0.0090 - val_roc_auc: 0.884

### Validation

In [ ]:
def predict_scores_in_chunks(df, model, dense_feature_cols, cat_feature_cols, label_col='label_engagement', chunk_size=300_000):
    scores = np.empty(len(df), dtype=np.float32)

    for start in range(0, len(df), chunk_size):
        end = min(start + chunk_size, len(df))

        chunk = df.iloc[start:end].copy()
        chunk = preprocess_pre_sample(chunk)
        chunk = clean_deep_features(chunk, dense_feature_cols, cat_feature_cols)

        ds = make_tf_dataset(chunk, dense_feature_cols, cat_feature_cols, label_col, shuffle=False)

        raw_preds = model.predict(ds, verbose=0)

        preds = tf.nn.sigmoid(raw_preds).numpy().reshape(-1)

        scores[start:end] = preds.astype(np.float32)
        print(f"Scored rows {start:,} to {end:,}")

    return scores

### Evaluation Functions - Ranker

In [ ]:
def evaluate_ranking(df, score_col, label_col="label_engagement", query_col="global_query_id",
                     ks=[10, 20, 50], mode="reranker_only"):
    eval_df = df.copy()

    if mode == "reranker_only":
        
        q_pos = eval_df.groupby(query_col)[label_col].sum()
        keep_qids = q_pos[q_pos > 0].index
        eval_df = eval_df[eval_df[query_col].isin(keep_qids)].copy()

    eval_df = eval_df.sort_values([query_col, score_col, "cg_rank"], ascending=[True, False, True])

    rows = []
    n_queries = eval_df[query_col].nunique()

    for k in ks:
        hit_list, recall_list, ndcg_list, mrr_list = [], [], [], []

        for _, g in eval_df.groupby(query_col, sort=False):
            labels = g[label_col].to_numpy()
            total_pos = labels.sum()

            if total_pos == 0:
                hit_list.append(0.0)
                recall_list.append(0.0)
                ndcg_list.append(0.0)
                mrr_list.append(0.0)
                continue

            topk = labels[:k]
            hits = topk.sum()

            hit = 1.0 if hits > 0 else 0.0
            recall = hits / total_pos

            discounts = 1.0 / np.log2(np.arange(2, len(topk) + 2))
            dcg = np.sum(topk * discounts)

            ideal_hits = int(min(total_pos, k))
            ideal_discounts = 1.0 / np.log2(np.arange(2, ideal_hits + 2))
            idcg = np.sum(ideal_discounts)
            ndcg = dcg / idcg if idcg > 0 else 0.0

            pos_idx = np.where(topk == 1)[0]
            mrr = 1.0 / (pos_idx[0] + 1) if len(pos_idx) > 0 else 0.0

            hit_list.append(hit)
            recall_list.append(recall)
            ndcg_list.append(ndcg)
            mrr_list.append(mrr)

        rows.append({
            "score_col": score_col,
            "mode": mode,
            "K": k,
            "n_queries": n_queries,
            "HitRate": np.mean(hit_list),
            "Recall": np.mean(recall_list),
            "NDCG": np.mean(ndcg_list),
            "MRR": np.mean(mrr_list),
        })

    return pd.DataFrame(rows)

In [ ]:
def add_lift(compare_df, baseline_score_col="cg_baseline_score", model_score_col="lr_score"):
    base = compare_df[compare_df["score_col"] == baseline_score_col].copy()
    model = compare_df[compare_df["score_col"] == model_score_col].copy()

    merged = model.merge(
        base,
        on=["mode", "K"],
        suffixes=("_model", "_baseline")
    )

    for metric in ["HitRate", "Recall", "NDCG", "MRR"]:
        merged[f"{metric}_lift_abs"] = merged[f"{metric}_model"] - merged[f"{metric}_baseline"]
        merged[f"{metric}_lift_pct"] = np.where(
            merged[f"{metric}_baseline"] > 0,
            100 * merged[f"{metric}_lift_abs"] / merged[f"{metric}_baseline"],
            np.nan
        )

    return merged

#### Evaluation Queries

In [ ]:
# validation data

val_scored = val_small[["global_query_id", "candidate_itemid", "cg_rank", "label_engagement"]].copy()
val_scored['dnn_v2_score'] = predict_scores_in_chunks(val_small, dnn_model_v2, dense_feature_cols, cat_feature_cols)

val_scored["cg_baseline_score"] = -val_scored["cg_rank"]

ks = [10, 20, 50, 100]

# reranker only
val_cg_rerank = evaluate_ranking(
    val_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

val_dnn_rerank = evaluate_ranking(
    val_scored,
    score_col="dnn_v2_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

# all queries
val_cg_all = evaluate_ranking(
    val_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

val_dnn_all = evaluate_ranking(
    val_scored,
    score_col="dnn_v2_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

# val dfs

val_compare_rerank = pd.concat([val_cg_rerank, val_dnn_rerank], ignore_index=True)
val_compare_all = pd.concat([val_cg_all, val_dnn_all], ignore_index=True)

Scored rows 0 to 300,000
Scored rows 300,000 to 600,000
Scored rows 600,000 to 900,000
Scored rows 900,000 to 1,200,000
Scored rows 1,200,000 to 1,500,000
Scored rows 1,500,000 to 1,800,000
Scored rows 1,800,000 to 2,100,000
Scored rows 2,100,000 to 2,400,000
Scored rows 2,400,000 to 2,700,000
Scored rows 2,700,000 to 3,000,000
Scored rows 3,000,000 to 3,010,291


In [ ]:
# test data - done on whole test data, can be sampled as well if needed.
# test_small = sample_query_groups(test_df, 10000, query_col='global_query_id', random_state=42)

test_scored = test_df[["global_query_id", "candidate_itemid", "cg_rank", "label_engagement"]].copy()
test_scored['dnn_v2_score'] = predict_scores_in_chunks(test_df, dnn_model_v2, dense_feature_cols, cat_feature_cols)

test_scored["cg_baseline_score"] = -test_scored["cg_rank"]

ks = [10, 20, 50, 100]

# reranker only
test_cg_rerank = evaluate_ranking(
    test_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

test_dnn_rerank = evaluate_ranking(
    test_scored,
    score_col="dnn_v2_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

# all queries
test_cg_all = evaluate_ranking(
    test_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

test_dnn_all = evaluate_ranking(
    test_scored,
    score_col="dnn_v2_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

# test dfs

test_compare_rerank = pd.concat([test_cg_rerank, test_dnn_rerank], ignore_index=True)
test_compare_all = pd.concat([test_cg_all, test_dnn_all], ignore_index=True)

Scored rows 0 to 300,000
Scored rows 300,000 to 600,000
Scored rows 600,000 to 900,000
Scored rows 900,000 to 1,200,000
Scored rows 1,200,000 to 1,500,000
Scored rows 1,500,000 to 1,800,000
Scored rows 1,800,000 to 2,100,000
Scored rows 2,100,000 to 2,400,000
Scored rows 2,400,000 to 2,700,000
Scored rows 2,700,000 to 3,000,000
Scored rows 3,000,000 to 3,300,000
Scored rows 3,300,000 to 3,600,000
Scored rows 3,600,000 to 3,900,000
Scored rows 3,900,000 to 4,200,000
Scored rows 4,200,000 to 4,500,000
Scored rows 4,500,000 to 4,800,000
Scored rows 4,800,000 to 5,100,000
Scored rows 5,100,000 to 5,400,000
Scored rows 5,400,000 to 5,700,000
Scored rows 5,700,000 to 6,000,000
Scored rows 6,000,000 to 6,300,000
Scored rows 6,300,000 to 6,600,000
Scored rows 6,600,000 to 6,900,000
Scored rows 6,900,000 to 7,200,000
Scored rows 7,200,000 to 7,500,000
Scored rows 7,500,000 to 7,800,000
Scored rows 7,800,000 to 8,100,000
Scored rows 8,100,000 to 8,400,000
Scored rows 8,400,000 to 8,700,000
Score

In [ ]:
val_lift_rerank = add_lift(val_compare_rerank, baseline_score_col="cg_baseline_score", model_score_col="dnn_v2_score")
val_lift_all = add_lift(val_compare_all, baseline_score_col="cg_baseline_score", model_score_col="dnn_v2_score")

val_lift_rerank[[
    "mode", "K",
    "NDCG_baseline", "NDCG_model", "NDCG_lift_abs", "NDCG_lift_pct",
    "HitRate_baseline", "HitRate_model", "HitRate_lift_abs", "HitRate_lift_pct"
]]

,mode,K,NDCG_baseline,NDCG_model,NDCG_lift_abs,NDCG_lift_pct,HitRate_baseline,HitRate_model,HitRate_lift_abs,HitRate_lift_pct
0,reranker_only,10,0.163966,0.427472,0.263506,160.707257,0.314721,0.639594,0.324873,103.225806
1,reranker_only,20,0.194895,0.446571,0.251676,129.134554,0.436548,0.710660,0.274112,62.790698
2,reranker_only,50,0.241261,0.458597,0.217337,90.083656,0.659898,0.766497,0.106599,16.153846
3,reranker_only,100,0.275667,0.464529,0.188862,68.510796,0.847716,0.791878,-0.055838,-6.586826


In [ ]:
test_lift_rerank = add_lift(test_compare_rerank, baseline_score_col="cg_baseline_score", model_score_col="dnn_v2_score")
test_lift_all = add_lift(test_compare_all, baseline_score_col="cg_baseline_score", model_score_col="dnn_v2_score")

test_lift_rerank[[
    "mode", "K",
    "NDCG_baseline", "NDCG_model", "NDCG_lift_abs", "NDCG_lift_pct",
    "HitRate_baseline", "HitRate_model", "HitRate_lift_abs", "HitRate_lift_pct"
]]

,mode,K,NDCG_baseline,NDCG_model,NDCG_lift_abs,NDCG_lift_pct,HitRate_baseline,HitRate_model,HitRate_lift_abs,HitRate_lift_pct
0,reranker_only,10,0.179708,0.406544,0.226836,126.225010,0.323293,0.646586,0.323293,100.000000
1,reranker_only,20,0.208113,0.430181,0.222068,106.705843,0.435743,0.724900,0.289157,66.359447
2,reranker_only,50,0.250840,0.450131,0.199291,79.449729,0.646586,0.813253,0.166667,25.776398
3,reranker_only,100,0.280788,0.458813,0.178025,63.401902,0.807229,0.861446,0.054217,6.716418


In [ ]:
## Interpretation

# the DNN v1 extends DNN v0 by including small embeddings of user history, day and hour of day features, 
# we dont expect huge improvements because these are small features and the main learning was about how to implement these embeddings cleanly
# the model performs similar to the earlier DNN v0 and logistic model and doesnt improve upon it
# This suggests that these smaller features add very limited contextual information when compared to CG score, rank and source features
# The next model will be learned with item level embeddings and is expected to perform better and later visitor level embeddings